# PLN — Coleta e pré-processamento de receitas do TudoGostoso

**Equipe:** Alana Cristina Andreazza, Beatriz Moresco Joaquim, Caio Abraão, Haiko Rudiger e Leticia Fruet

O notebook realiza duas etapas principais:

1. coleta das receitas do TudoGostoso;
2. pré-processamento textual seguindo a estrutura apresentada pelo professor.

A base mantém os campos estruturados de cada receita e cria também um único campo textual para aplicação do pipeline de PLN.

## 1. Preparação do ambiente

Para a coleta são utilizados `requests` e `BeautifulSoup`.  
Para o pré-processamento, seguimos as bibliotecas utilizadas no notebook do professor:

- **NLTK**: tokenização, stopwords e stemming;
- **spaCy**: lematização;
- **Pandas**: organização dos dados.

In [ ]:
%pip -q install requests beautifulsoup4 pandas nltk spacy

In [ ]:
import re
import subprocess
import sys
import time

import nltk
import pandas as pd
import requests
import spacy

from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from nltk.tokenize import word_tokenize
from urllib.parse import urljoin

for recurso in ["punkt", "punkt_tab", "stopwords", "rslp"]:
    nltk.download(recurso, quiet=True)

try:
    nlp = spacy.load("pt_core_news_sm")
except OSError:
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", "pt_core_news_sm"],
        check=True,
    )
    nlp = spacy.load("pt_core_news_sm")

print("Ambiente preparado com sucesso.")

Ambiente preparado com sucesso.


## 2. Categorias utilizadas

Foram mantidas as categorias definidas pelo grupo para obter variedade de receitas.

In [ ]:
BASE_URL = "https://www.tudogostoso.com.br"

CATEGORIAS = [
    {"categoria": "BOLOS E TORTAS", "url": "https://www.tudogostoso.com.br/categorias/1000-bolos-e-tortas-doces"},
    {"categoria": "CARNES", "url": "https://www.tudogostoso.com.br/categorias/1004-carnes"},
    {"categoria": "AVES", "url": "https://www.tudogostoso.com.br/categorias/1009-aves"},
    {"categoria": "PEIXES E FRUTOS DO MAR", "url": "https://www.tudogostoso.com.br/categorias/1014-peixes-e-frutos-do-mar"},
    {"categoria": "SALADAS E MOLHOS", "url": "https://www.tudogostoso.com.br/categorias/1023-saladas-molhos-e-acompanhamentos"},
    {"categoria": "SOPAS", "url": "https://www.tudogostoso.com.br/categorias/1027-sopas"},
    {"categoria": "MASSAS", "url": "https://www.tudogostoso.com.br/categorias/1028-massas"},
    {"categoria": "BEBIDAS", "url": "https://www.tudogostoso.com.br/categorias/1032-bebidas"},
    {"categoria": "DOCES E SOBREMESAS", "url": "https://www.tudogostoso.com.br/categorias/1037-doces-e-sobremesas"},
    {"categoria": "LANCHES", "url": "https://www.tudogostoso.com.br/categorias/1044-lanches"},
    {"categoria": "ALIMENTAÇÃO SAUDÁVEL", "url": "https://www.tudogostoso.com.br/categorias/1334-alimentacao-saudavel"},
]

LIMITE_RECEITAS_POR_CATEGORIA = 15

## 3. Acesso às páginas

A estrutura anterior dependia de caminhos CSS longos e de posições como `nth-child`.  
Isso tornava a coleta sensível a pequenas mudanças no HTML.

Nesta versão, a extração usa principalmente classes e seções semânticas da página.  
A versão AMP oficial da mesma receita é usada somente como apoio para campos que podem não estar presentes de forma estável no HTML principal, como avaliação, dificuldade e custo.

In [ ]:
sessao = requests.Session()
sessao.headers.update({
    "User-Agent": "FURB-PLN-AvaliacaoPratica1/1.0 (uso academico)",
    "Accept-Language": "pt-BR,pt;q=0.9",
})


def obter_soup(url):
    try:
        resposta = sessao.get(url, timeout=20)
        resposta.raise_for_status()
        return BeautifulSoup(resposta.content, "html.parser")
    except requests.RequestException as erro:
        print(f"Erro ao acessar {url}: {erro}")
        return None


def url_amp(url):
    return url.replace(
        "https://www.tudogostoso.com.br/",
        "https://amp.tudogostoso.com.br/"
    )

## 4. Localização das receitas nas categorias

Primeiro reunimos os links de todas as categorias.

A URL é usada como identificador da receita. Se a mesma URL aparecer em duas ou mais categorias, ela permanece uma única vez e o campo `Categorias` recebe todas as categorias correspondentes.

In [ ]:
SELETOR_TITULOS_RECEITAS = (
    "#main-content div.mg-container.gd-2-cols div.left-col "
    "section.u-margin-top.u-padding-top.u-padding-bottom "
    "div.card-content h2 a"
)


def extrair_receitas_categoria(categoria, url_categoria, limite=None):
    soup = obter_soup(url_categoria)

    if soup is None:
        return []

    elementos = soup.select(SELETOR_TITULOS_RECEITAS)

    if limite is not None:
        elementos = elementos[:limite]

    return [
        {
            "Título": elemento.get_text(" ", strip=True),
            "URL da receita": urljoin(
                BASE_URL,
                elemento.get("href", "")
            ),
            "Categoria": categoria,
        }
        for elemento in elementos
        if elemento.get("href")
    ]


receitas_por_url = {}

for categoria in CATEGORIAS:
    encontradas = extrair_receitas_categoria(
        categoria["categoria"],
        categoria["url"],
        LIMITE_RECEITAS_POR_CATEGORIA,
    )

    for receita in encontradas:
        url = receita["URL da receita"]

        if url not in receitas_por_url:
            receitas_por_url[url] = {
                "Título": receita["Título"],
                "URL da receita": url,
                "Categorias": [],
            }

        if receita["Categoria"] not in receitas_por_url[url]["Categorias"]:
            receitas_por_url[url]["Categorias"].append(
                receita["Categoria"]
            )

print("Receitas únicas encontradas:", len(receitas_por_url))

Receitas únicas encontradas: 159


## 5. Extração dos dados da receita

- ingredientes são procurados dentro de `section.recipe-ingredients`;
- utensílios dentro de `section.recipe-equipments`;
- etapas são localizadas pelos IDs que começam com `recipe-step`;
- porções são obtidas pelo título da seção de ingredientes;
- avaliação, quantidade de avaliações, dificuldade e custo possuem fallback pela página AMP oficial.

In [ ]:
def texto_elemento(elemento):
    return elemento.get_text(" ", strip=True) if elemento else None


def limpar_texto_utensilio(texto):
    if not texto:
        return None

    texto = re.sub(r"\bComprar\b", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto or None


def extrair_porcoes(soup):
    secao = soup.select_one("section.recipe-ingredients")

    if secao:
        titulo = secao.find(["h1", "h2", "h3"])
        if titulo:
            texto = titulo.get_text(" ", strip=True)
            resultado = re.search(
                r"(\d+\s+porç(?:ão|ões))",
                texto,
                flags=re.IGNORECASE,
            )
            if resultado:
                return resultado.group(1)

    return None


def extrair_ingredientes(soup):
    secao = soup.select_one("section.recipe-ingredients")

    if not secao:
        return []

    ingredientes = []

    for item in secao.select("li"):
        texto = item.get_text(" ", strip=True)

        if texto:
            ingredientes.append(texto)

    return ingredientes


def extrair_utensilios(soup):
    secao = soup.select_one("section.recipe-equipments")

    if not secao:
        return []

    utensilios = []

    for item in secao.select("li"):
        texto = limpar_texto_utensilio(
            item.get_text(" ", strip=True)
        )

        if texto:
            utensilios.append(texto)

    return utensilios


def extrair_etapas(soup):
    etapas = []

    for elemento in soup.select('[id^="recipe-step"] p'):
        texto = elemento.get_text(" ", strip=True)

        if texto and texto not in etapas:
            etapas.append(texto)

    return etapas


def extrair_tempo(soup):
    # Primeiro tenta o bloco de informações da receita.
    for bloco in soup.select(".recipe-info > div"):
        texto = bloco.get_text(" ", strip=True)
        if "min" in texto.lower():
            return texto

    # Fallback para o texto exibido na área "Modo de preparo".
    texto_pagina = soup.get_text(" ", strip=True)
    resultado = re.search(
        r"Modo de preparo\s*:\s*(\d+\s*min)",
        texto_pagina,
        flags=re.IGNORECASE,
    )

    return resultado.group(1) if resultado else None


def extrair_metadados_amp(soup_amp):
    dados = {
        "Nível de dificuldade": None,
        "Custo": None,
        "Avaliação": None,
        "Quantidade de avaliações": None,
    }

    if soup_amp is None:
        return dados

    texto = soup_amp.get_text(" ", strip=True)

    avaliacao = re.search(
        r"(\d+(?:[.,]\d+)?)\s*/\s*5",
        texto,
        flags=re.IGNORECASE,
    )
    if avaliacao:
        dados["Avaliação"] = avaliacao.group(1).replace(",", ".")

    quantidade = re.search(
        r"\(\s*([\d.,]+(?:\s*[KkMm])?)\s+avaliaç(?:ão|ões)\s*\)",
        texto,
        flags=re.IGNORECASE,
    )
    if quantidade:
        dados["Quantidade de avaliações"] = quantidade.group(1)

    dificuldade = re.search(
        r"\b(Muito fácil|Fácil|Médio|Média|Difícil|Muito difícil)\b",
        texto,
        flags=re.IGNORECASE,
    )
    if dificuldade:
        dados["Nível de dificuldade"] = dificuldade.group(1)

    custo = re.search(
        r"\b(Custo\s+(?:baixo|médio|alto))\b",
        texto,
        flags=re.IGNORECASE,
    )
    if custo:
        dados["Custo"] = custo.group(1)

    return dados


def extrair_dados_receita(receita):
    url = receita["URL da receita"]

    soup = obter_soup(url)

    if soup is None:
        return None

    # Campos principais do HTML da receita.
    porcoes = extrair_porcoes(soup)
    ingredientes = extrair_ingredientes(soup)
    utensilios = extrair_utensilios(soup)
    etapas = extrair_etapas(soup)
    tempo = extrair_tempo(soup)

    # Alguns campos podem estar ausentes no HTML principal.
    soup_amp = obter_soup(url_amp(url))
    metadados_amp = extrair_metadados_amp(soup_amp)

    return {
        "Categorias": receita["Categorias"],
        "Título": receita["Título"],
        "URL da receita": url,
        "Tempo de preparo": tempo,
        "Nível de dificuldade": metadados_amp["Nível de dificuldade"],
        "Custo": metadados_amp["Custo"],
        "Porções": porcoes,
        "Ingredientes": ingredientes,
        "Utensílios": utensilios,
        "Etapas do preparo": etapas,
        "Avaliação": metadados_amp["Avaliação"],
        "Quantidade de avaliações": metadados_amp["Quantidade de avaliações"],
    }

## 6. Executar a coleta

Cada URL é processada somente uma vez, mesmo que a receita pertença a várias categorias.

In [ ]:
resultados = []

for numero, receita in enumerate(receitas_por_url.values(), start=1):
    print(f"{numero}. Processando: {receita['Título']}")

    dados = extrair_dados_receita(receita)

    if dados is not None:
        resultados.append(dados)

    time.sleep(1)

df_receitas = pd.DataFrame(resultados)

print("\nReceitas armazenadas:", len(df_receitas))
display(df_receitas.head(10))

1. Processando: Bolo de milho de lata no liquidificador
2. Processando: Bolo de fubá simples
3. Processando: Brownie simples e rápido
4. Processando: Bolo simples
5. Processando: Bolo de caneca micro-ondas
6. Processando: Bolo de laranja
7. Processando: Bolo pão-de-ló tradicional
8. Processando: Bolo de laranja de liquidificador
9. Processando: Bolo de iogurte
10. Processando: Pavê de chocolate
11. Processando: Bolo de fubá cremoso
12. Processando: Bolo flocão
13. Processando: Bolo de liquidificador
14. Processando: Bolo de coco gelado
15. Processando: Bolo de maçã de liquidificador - o melhor do mundo
16. Processando: Carne de panela de pressão
17. Processando: Rocambole de carne moída maravilhoso
18. Processando: Yakissoba da casa
19. Processando: Strogonoff de carne
20. Processando: Bife à milanesa
21. Processando: Escondidinho de carne-seca
22. Processando: Almôndegas
23. Processando: Arroz de forno à parmegiana
24. Processando: Quibe fácil
25. Processando: Costela na panela de pre

,Categorias,Título,URL da receita,Tempo de preparo,Nível de dificuldade,Custo,Porções,Ingredientes,Utensílios,Etapas do preparo,Avaliação,Quantidade de avaliações
0,[BOLOS E TORTAS],Bolo de milho de lata no liquidificador,https://www.tudogostoso.com.br/receita/143393-bolo-de-milho-de-lata-no-liquidificador.html,50min,Muito fácil,Custo baixo,1 porção,"[1 lata de milho (sem o líquido), 1 lata de leite (medida da lata de milho), 1 lata de açúcar (m...","[Liquidificador, Pincel de silicone, Espátula de silicone, Formo de bolo, Forno elétrico]","[Escorra o milho e use a própria lata para as medidas., Unte e enfarinhe uma forma de bolo com f...",4.8,1768
1,[BOLOS E TORTAS],Bolo de fubá simples,https://www.tudogostoso.com.br/receita/21560-bolo-de-fuba-simples.html,45min,Fácil,Custo baixo,8 porções,"[3 ovos, 2 xícaras (chá) de açúcar, 2 xícaras (chá) de fubá, 3 colheres (sopa) rasas de farinha ...","[Liquidificador, Copo medidor, Formo de bolo, Prato de sobremesa]","[Bata todos os ingredientes no liquidificador., Coloque em uma forma untada e enfarinhada., Leve...",4.4,1021
2,[BOLOS E TORTAS],Brownie simples e rápido,https://www.tudogostoso.com.br/receita/306823-brownie-simples-e-rapido.html,30min,Fácil,Custo baixo,10 porções,"[5 colheres de manteiga, 3 ovos, 3 xicara de achocolatado, 6 colheres de açúcar, 12 colheres de ...","[Bowl, Espatula, Forma, Fouet, Prato de sobremesa]","[Derreta a manteiga e reserve, Enquanto derrete a manteiga, misture os 3 ovos e a açúcar e mistu...",3.8,623
3,[BOLOS E TORTAS],Bolo simples,https://www.tudogostoso.com.br/receita/29124-bolo-simples.html,40min,Muito fácil,Custo baixo,12 porções,"[2 xícaras (chá) de açúcar, 3 xícaras (chá) de farinha de trigo, 4 colheres (sopa) de margarina,...","[Batedeira, Bowl, Espátula para bolo, Formo de bolo, Prato de sobremesa]","[Bata as claras em neve e reserve., Misture as gemas, a margarina e o açúcar até obter uma massa...",4.7,3787
4,[BOLOS E TORTAS],Bolo de caneca micro-ondas,https://www.tudogostoso.com.br/receita/60429-bolo-de-caneca-micro-ondas.html,10min,Fácil,Custo baixo,1 porção,"[1 ovo, 2 colheres (sopa) de achocolatado em pó, 3 colheres (sopa) rasas de açúcar, 4 colheres (...","[Caneca, Panela, Colher para sobremesa, Luva térmica, Micro-ondas]","[Coloque todos os ingredientes dentro de uma caneca de aproximadamente 300 ml ou mais., Mexa até...",4.4,1796
5,[BOLOS E TORTAS],Bolo de laranja,https://www.tudogostoso.com.br/receita/13953-bolo-de-laranja.html,40min,fácil,Custo baixo,12 porções,"[4 ovos, 2 xícaras (chá) de açúcar, 1 xícara (chá) de óleo, suco de 2 laranjas, casca de 1 laran...","[Liquidificador, Espátula para bolo, Formo de bolo]","[Bata no liquidificador os ovos, o açúcar, o óleo, o suco e a casca da laranja., Passe para uma ...",4.6,742
6,[BOLOS E TORTAS],Bolo pão-de-ló tradicional,https://www.tudogostoso.com.br/receita/142803-bolo-pao-de-lo-tradicional.html,30min,Fácil,Custo baixo,30 porções,"[1 xícara de leite quente, 3 xícaras de farinha de trigo, 2 xícaras de açúcar, 3 gemas, 3 claras...","[Batedeira, Copo medidor, Espátula para bolo, Formo de bolo, Prato de sobremesa]","[Bata as claras em neve até ficar bem consistente., Acrescente o açúcar e as gemas, bata bem., J...",4,237
7,[BOLOS E TORTAS],Bolo de laranja de liquidificador,https://www.tudogostoso.com.br/receita/85429-bolo-de-laranja-de-liquidificador.html,None,Fácil,Custo baixo,10 porções,"[3 ovos, Suco de 2 laranjas, 1 xícara (chá) de óleo, 2 xícaras (chá) de açúcar, 3 xícaras (chá) ...","[Liquidificador, Bowl, Formo de bolo, Fouet]","[Bata no liquidificador os 4 primeiros ingredientes (exceto farinha e fermento), despeje em uma ...",4.7,957
8,[BOLOS E TORTAS],Bolo de iogurte,https://www.tudogostoso.com.br/receita/55080-bolo-de-iogurte.html,50min,Fácil,Custo baixo,12 porções,"[4 ovos, 1 copo de iogurte natural, ½ copo de óleo, 2 copos de açúcar, 2 copos de farinha de tri...","[Forma de pudim, Liquidificador, Copo medidor, Espátula de silicone]","[Bater tudo no liquidificador., Untar uma

## 7. Verificação rápida da coleta

Esta conferência é curta e faz parte da validação do resultado final, sem manter toda a estrutura de logs e diagnóstico da versão anterior.

In [ ]:
if df_receitas.empty:
    print("Nenhuma receita foi coletada.")
else:
    print("URLs duplicadas:", df_receitas["URL da receita"].duplicated().sum())

    resumo_campos = pd.DataFrame({
        "Campo": [
            "Porções",
            "Ingredientes",
            "Utensílios",
            "Etapas do preparo",
            "Avaliação",
        ],
        "Registros preenchidos": [
            df_receitas["Porções"].notna().sum(),
            df_receitas["Ingredientes"].apply(bool).sum(),
            df_receitas["Utensílios"].apply(bool).sum(),
            df_receitas["Etapas do preparo"].apply(bool).sum(),
            df_receitas["Avaliação"].notna().sum(),
        ],
    })

    display(resumo_campos)

    url_teste = (
        "https://www.tudogostoso.com.br/receita/"
        "143393-bolo-de-milho-de-lata-no-liquidificador.html"
    )

    teste = df_receitas[df_receitas["URL da receita"] == url_teste]

    if not teste.empty:
        display(
            teste[
                [
                    "Título",
                    "Categorias",
                    "Porções",
                    "Ingredientes",
                    "Utensílios",
                    "Etapas do preparo",
                    "Avaliação",
                    "Quantidade de avaliações",
                ]
            ]
        )

URLs duplicadas: 0


,Campo,Registros preenchidos
0,Porções,156
1,Ingredientes,159
2,Utensílios,112
3,Etapas do preparo,159
4,Avaliação,159


,Título,Categorias,Porções,Ingredientes,Utensílios,Etapas do preparo,Avaliação,Quantidade de avaliações
0,Bolo de milho de lata no liquidificador,[BOLOS E TORTAS],1 porção,"[1 lata de milho (sem o líquido), 1 lata de leite (medida da lata de milho), 1 lata de açúcar (m...","[Liquidificador, Pincel de silicone, Espátula de silicone, Formo de bolo, Forno elétrico]","[Escorra o milho e use a própria lata para as medidas., Unte e enfarinhe uma forma de bolo com f...",4.8,1768


## 8. Campo textual único para PLN

Os campos estruturados continuam separados.

Além deles, criamos `Texto para PLN` reunindo somente:

- porções;
- ingredientes;
- utensílios;
- etapas do preparo.

Esse será o campo usado uma única vez pelo pipeline de pré-processamento.

In [ ]:
def lista_para_texto(valor):
    if isinstance(valor, list):
        return " ".join(str(item) for item in valor if item)
    return "" if pd.isna(valor) else str(valor)


def montar_texto_pln(linha):
    partes = [
        linha["Porções"] or "",
        lista_para_texto(linha["Ingredientes"]),
        lista_para_texto(linha["Utensílios"]),
        lista_para_texto(linha["Etapas do preparo"]),
    ]

    return " ".join(
        parte.strip()
        for parte in partes
        if parte and parte.strip()
    )


df_receitas["Texto para PLN"] = df_receitas.apply(
    montar_texto_pln,
    axis=1,
)

display(
    df_receitas[
        [
            "Título",
            "Porções",
            "Ingredientes",
            "Utensílios",
            "Etapas do preparo",
            "Texto para PLN",
        ]
    ].head(3)
)

,Título,Porções,Ingredientes,Utensílios,Etapas do preparo,Texto para PLN
0,Bolo de milho de lata no liquidificador,1 porção,"[1 lata de milho (sem o líquido), 1 lata de leite (medida da lata de milho), 1 lata de açúcar (m...","[Liquidificador, Pincel de silicone, Espátula de silicone, Formo de bolo, Forno elétrico]","[Escorra o milho e use a própria lata para as medidas., Unte e enfarinhe uma forma de bolo com f...",1 porção 1 lata de milho (sem o líquido) 1 lata de leite (medida da lata de milho) 1 lata de açú...
1,Bolo de fubá simples,8 porções,"[3 ovos, 2 xícaras (chá) de açúcar, 2 xícaras (chá) de fubá, 3 colheres (sopa) rasas de farinha ...","[Liquidificador, Copo medidor, Formo de bolo, Prato de sobremesa]","[Bata todos os ingredientes no liquidificador., Coloque em uma forma untada e enfarinhada., Leve...",8 porções 3 ovos 2 xícaras (chá) de açúcar 2 xícaras (chá) de fubá 3 colheres (sopa) rasas de fa...
2,Brownie simples e rápido,10 porções,"[5 colheres de manteiga, 3 ovos, 3 xicara de achocolatado, 6 colheres de açúcar, 12 colheres de ...","[Bowl, Espatula, Forma, Fouet, Prato de sobremesa]","[Derreta a manteiga e reserve, Enquanto derrete a manteiga, misture os 3 ovos e a açúcar e mistu...",10 porções 5 colheres de manteiga 3 ovos 3 xicara de achocolatado 6 colheres de açúcar 12 colher...


# Parte 2 — Pré-processamento

A partir daqui, a organização segue diretamente a lógica do notebook do professor:

**tokenização → normalização → remoção de stopwords → lematização → stemming**.

O processamento é feito apenas sobre `Texto para PLN`.

## 9. Tokenização

In [ ]:
tokens = [
    word_tokenize(texto, language="portuguese")
    for texto in df_receitas["Texto para PLN"]
]

df_receitas["tokens"] = tokens

display(df_receitas[["Título", "Texto para PLN", "tokens"]].head(3))

,Título,Texto para PLN,tokens
0,Bolo de milho de lata no liquidificador,1 porção 1 lata de milho (sem o líquido) 1 lata de leite (medida da lata de milho) 1 lata de açú...,"[1, porção, 1, lata, de, milho, (, sem, o, líquido, ), 1, lata, de, leite, (, medida, da, lata, ..."
1,Bolo de fubá simples,8 porções 3 ovos 2 xícaras (chá) de açúcar 2 xícaras (chá) de fubá 3 colheres (sopa) rasas de fa...,"[8, porções, 3, ovos, 2, xícaras, (, chá, ), de, açúcar, 2, xícaras, (, chá, ), de, fubá, 3, col..."
2,Brownie simples e rápido,10 porções 5 colheres de manteiga 3 ovos 3 xicara de achocolatado 6 colheres de açúcar 12 colher...,"[10, porções, 5, colheres, de, manteiga, 3, ovos, 3, xicara, de, achocolatado, 6, colheres, de, ..."


## 10. Normalização

Assim como no exemplo do professor, os tokens são convertidos com `casefold()` e permanecem apenas tokens alfabéticos.

Essa etapa também remove a pontuação da representação normalizada.

In [ ]:
def normalizar_tokens(tokens_da_receita):
    return [
        token.casefold()
        for token in tokens_da_receita
        if token.isalpha()
    ]


df_receitas["tokens_normalizados"] = (
    df_receitas["tokens"]
    .apply(normalizar_tokens)
)

display(
    df_receitas[
        ["Título", "tokens", "tokens_normalizados"]
    ].head(3)
)

,Título,tokens,tokens_normalizados
0,Bolo de milho de lata no liquidificador,"[1, porção, 1, lata, de, milho, (, sem, o, líquido, ), 1, lata, de, leite, (, medida, da, lata, ...","[porção, lata, de, milho, sem, o, líquido, lata, de, leite, medida, da, lata, de, milho, lata, d..."
1,Bolo de fubá simples,"[8, porções, 3, ovos, 2, xícaras, (, chá, ), de, açúcar, 2, xícaras, (, chá, ), de, fubá, 3, col...","[porções, ovos, xícaras, chá, de, açúcar, xícaras, chá, de, fubá, colheres, sopa, rasas, de, far..."
2,Brownie simples e rápido,"[10, porções, 5, colheres, de, manteiga, 3, ovos, 3, xicara, de, achocolatado, 6, colheres, de, ...","[porções, colheres, de, manteiga, ovos, xicara, de, achocolatado, colheres, de, açúcar, colheres..."


## 11. Remoção de stopwords

In [ ]:
STOPWORDS_PT = set(stopwords.words("portuguese"))

# Em receitas, "não" e "sem" podem alterar o sentido da instrução.
STOPWORDS_RECEITAS = STOPWORDS_PT - {"não", "nao", "sem"}

def remover_stopwords(tokens_da_receita):
    return [
        token
        for token in tokens_da_receita
        if token not in STOPWORDS_RECEITAS
    ]

df_receitas["tokens_sem_stopwords"] = (
    df_receitas["tokens_normalizados"]
    .apply(remover_stopwords)
)

display(
    df_receitas[
        ["Título", "tokens_normalizados", "tokens_sem_stopwords"]
    ].head(3)
)

,Título,tokens_normalizados,tokens_sem_stopwords
0,Bolo de milho de lata no liquidificador,"[porção, lata, de, milho, sem, o, líquido, lata, de, leite, medida, da, lata, de, milho, lata, d...","[porção, lata, milho, sem, líquido, lata, leite, medida, lata, milho, lata, açúcar, medida, lata..."
1,Bolo de fubá simples,"[porções, ovos, xícaras, chá, de, açúcar, xícaras, chá, de, fubá, colheres, sopa, rasas, de, far...","[porções, ovos, xícaras, chá, açúcar, xícaras, chá, fubá, colheres, sopa, rasas, farinha, trigo,..."
2,Brownie simples e rápido,"[porções, colheres, de, manteiga, ovos, xicara, de, achocolatado, colheres, de, açúcar, colheres...","[porções, colheres, manteiga, ovos, xicara, achocolatado, colheres, açúcar, colheres, farinha, t..."


## 12. Lematização

In [ ]:
def lematizar(tokens_da_receita):
    texto_processavel = " ".join(tokens_da_receita)
    documento = nlp(texto_processavel)
    return [token.lemma_ for token in documento]


df_receitas["lemas"] = (
    df_receitas["tokens_sem_stopwords"]
    .apply(lematizar)
)

display(
    df_receitas[
        ["Título", "tokens_sem_stopwords", "lemas"]
    ].head(3)
)

,Título,tokens_sem_stopwords,lemas
0,Bolo de milho de lata no liquidificador,"[porção, lata, milho, sem, líquido, lata, leite, medida, lata, milho, lata, açúcar, medida, lata...","[porção, lato, milho, sem, líquido, lato, leite, medida, lato, milho, lato, açúcar, medida, lato..."
1,Bolo de fubá simples,"[porções, ovos, xícaras, chá, açúcar, xícaras, chá, fubá, colheres, sopa, rasas, farinha, trigo,...","[porção, ovo, xícara, chá, açúcar, xícara, chá, fubá, colher, sopa, raso, Farinha, trigo, copo, ..."
2,Brownie simples e rápido,"[porções, colheres, manteiga, ovos, xicara, achocolatado, colheres, açúcar, colheres, farinha, t...","[porção, colher, manteiga, ovo, xicara, achocolatar, colher, açúcar, colher, farinho, trigo, bow..."


## 13. Stemming

In [ ]:
stemmer = RSLPStemmer()


def aplicar_stemming(tokens_da_receita):
    return [
        stemmer.stem(token)
        for token in tokens_da_receita
    ]


df_receitas["stems"] = (
    df_receitas["tokens_sem_stopwords"]
    .apply(aplicar_stemming)
)

display(
    df_receitas[
        ["Título", "tokens_sem_stopwords", "stems"]
    ].head(3)
)

,Título,tokens_sem_stopwords,stems
0,Bolo de milho de lata no liquidificador,"[porção, lata, milho, sem, líquido, lata, leite, medida, lata, milho, lata, açúcar, medida, lata...","[porç, lat, milh, sem, líqu, lat, leit, med, lat, milh, lat, açúc, med, lat, milh, lat, floc, mi..."
1,Bolo de fubá simples,"[porções, ovos, xícaras, chá, açúcar, xícaras, chá, fubá, colheres, sopa, rasas, farinha, trigo,...","[porç, ovo, xíc, chá, açúc, xíc, chá, fub, colh, sop, ras, far, trig, cop, americ, óle, cop, lei..."
2,Brownie simples e rápido,"[porções, colheres, manteiga, ovos, xicara, achocolatado, colheres, açúcar, colheres, farinha, t...","[porç, colh, manteig, ovo, xic, achocolat, colh, açúc, colh, far, trig, bowl, espatul, form, fou..."


## 14. Pipeline completo com Pandas

Esta função reúne as mesmas etapas, seguindo o formato demonstrado pelo professor.

In [ ]:
def processar_texto(texto):
    tokens_locais = word_tokenize(
        texto,
        language="portuguese"
    )
    normalizados_locais = normalizar_tokens(tokens_locais)
    sem_stopwords_locais = remover_stopwords(
        normalizados_locais
    )

    return pd.Series({
        "tokens_pipeline": tokens_locais,
        "tokens_normalizados_pipeline": normalizados_locais,
        "tokens_sem_stopwords_pipeline": sem_stopwords_locais,
        "lemas_pipeline": lematizar(sem_stopwords_locais),
        "stems_pipeline": aplicar_stemming(sem_stopwords_locais),
    })


colunas_processadas = (
    df_receitas["Texto para PLN"]
    .apply(processar_texto)
)

df_processado = pd.concat(
    [df_receitas, colunas_processadas],
    axis=1,
)

pd.set_option("display.max_colwidth", 100)
display(df_processado.head())

,Categorias,Título,URL da receita,Tempo de preparo,Nível de dificuldade,Custo,Porções,Ingredientes,Utensílios,Etapas do preparo,...,tokens,tokens_normalizados,tokens_sem_stopwords,lemas,stems,tokens_pipeline,tokens_normalizados_pipeline,tokens_sem_stopwords_pipeline,lemas_pipeline,stems_pipeline
0,[BOLOS E TORTAS],Bolo de milho de lata no liquidificador,https://www.tudogostoso.com.br/receita/143393-bolo-de-milho-de-lata-no-liquidificador.html,50min,Muito fácil,Custo baixo,1 porção,"[1 lata de milho (sem o líquido), 1 lata de leite (medida da lata de milho), 1 lata de açúcar (m...","[Liquidificador, Pincel de silicone, Espátula de silicone, Formo de bolo, Forno elétrico]","[Escorra o milho e use a própria lata para as medidas., Unte e enfarinhe uma forma de bolo com f...",...,"[1, porção, 1, lata, de, milho, (, sem, o, líquido, ), 1, lata, de, leite, (, medida, da, lata, ...","[porção, lata, de, milho, sem, o, líquido, lata, de, leite, medida, da, lata, de, milho, lata, d...","[porção, lata, milho, sem, líquido, lata, leite, medida, lata, milho, lata, açúcar, medida, lata...","[porção, lato, milho, sem, líquido, lato, leite, medida, lato, milho, lato, açúcar, medida, lato...","[porç, lat, milh, sem, líqu, lat, leit, med, lat, milh, lat, açúc, med, lat, milh, lat, floc, mi...","[1, porção, 1, lata, de, milho, (, sem, o, líquido, ), 1, lata, de, leite, (, medida, da, lata, ...","[porção, lata, de, milho, sem, o, líquido, lata, de, leite, medida, da, lata, de, milho, lata, d...","[porção, lata, milho, sem, líquido, lata, leite, medida, lata, milho, lata, açúcar, medida, lata...","[porção, lato, milho, sem, líquido, lato, leite, medida, lato, milho, lato, açúcar, medida, lato...","[porç, lat, milh, sem, líqu, lat, leit, med, lat, milh, lat, açúc, med, lat, milh, lat, floc, mi..."
1,[BOLOS E TORTAS],Bolo de fubá simples,https://www.tudogostoso.com.br/receita/21560-bolo-de-fuba-simples.html,45min,Fácil,Custo baixo,8 porções,"[3 ovos, 2 xícaras (chá) de açúcar, 2 xícaras (chá) de fubá, 3 colheres (sopa) rasas de farinha ...","[Liquidificador, Copo medidor, Formo de bolo, Prato de sobremesa]","[Bata todos os ingredientes no liquidificador., Coloque em uma forma untada e enfarinhada., Leve...",...,"[8, porções, 3, ovos, 2, xícaras, (, chá, ), de, açúcar, 2, xícaras, (, chá, ), de, fubá, 3, col...","[porções, ovos, xícaras, chá, de, açúcar, xícaras, chá, de, fubá, colheres, sopa, rasas, de, far...","[porções, ovos, xícaras, chá, açúcar, xícaras, chá, fubá, colheres, sopa, rasas, farinha, trigo,...","[porção, ovo, xícara, chá, açúcar, xícara, chá, fubá, colher, sopa, raso, Farinha, trigo, copo, ...","[porç, ovo, xíc, chá, açúc, xíc, chá, fub, colh, sop, ras, far, trig, cop, americ, óle, cop, lei...","[8, porções, 3, ovos, 2, xícaras, (, chá, ), de, açúcar, 2, xícaras, (, chá, ), de, fubá, 3, col...","[porções, ovos, xícaras, chá, de, açúcar, xícaras, chá, de, fubá, colheres, sopa, rasas, de, far...","[porções, ovos, xícaras, chá, açúcar, xícaras, chá, fubá, colheres, sopa, rasas, farinha, trigo,...","[porção, ovo, xícara, chá, açúcar, xícara, chá, fubá, colher, sopa, raso, Farinha, trigo, copo, ...","[porç, ovo, xíc, chá, açúc, xíc, chá, fub, colh, sop, ras, far, trig, cop, americ, óle, cop, lei..."
2,[BOLOS E TORTAS],Brownie simples e rápido,https://www.tudogostoso.com.br/receita/306823-brownie-simples-e-rapido.html,30min,Fácil,Custo baixo,10 porções,"[5 colheres de manteiga, 3 ovos, 3 xicara de achocolatado, 6 colheres de açúcar, 12 colheres de ...","[Bowl, Espatula, Forma, Fouet, Prato de sobremesa]","[Derreta a manteiga e reserve, Enquanto derrete a manteiga, misture os 3 ovos e a açúcar e mistu...",...,"[10, porções, 5, colheres, de, manteiga, 3, ovos, 3, xicara, de, achocolatado, 6, colheres, de, ...","[porções, colheres, de, manteiga, ovos, xicara, de, achocolatado, colheres, de, açúcar, colheres...","[porções, colheres, manteiga, ovos, xicara, achocolatado, colheres, açúcar, colheres, farinha, t...","[porção, colher, manteiga, ovo

## 15. Exportação

A saída contém:

- campos originais da receita;
- lista de categorias;
- campo textual consolidado;
- representações geradas pelo pré-processamento.

In [ ]:
df_processado.to_csv(
    "receitas_tudogostoso_processadas.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Arquivo gerado: receitas_tudogostoso_processadas.csv")

Arquivo gerado: receitas_tudogostoso_processadas.csv
